# 計算每個文檔的token數量

In [1]:
import os
import json
import sys
from pathlib import Path
from transformers import AutoTokenizer
from tqdm import tqdm


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "Legal Case Retrieval" / "lcr" / "task1_paths.py").exists():
            return path
    raise FileNotFoundError("Cannot locate repo root from current working directory")


repo_root = find_repo_root(Path.cwd())
package_root = repo_root / "Legal Case Retrieval"
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

from lcr.task1_paths import get_task1_dir, get_task1_year

task1_dir = Path(get_task1_dir())
task1_year = get_task1_year()
print(f"Using Task1 year: {task1_year}, dir: {task1_dir}")

# ===== 設定多個資料夾路徑 =====
base_folders = [
    task1_dir / "processed",
    task1_dir / "processed_new",
    task1_dir / "summary",
]

# 載入 tokenizer
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

for dataset_dir in base_folders:
    if not dataset_dir.exists():
        print(f"Skip missing directory: {dataset_dir}")
        continue

    output_json = dataset_dir / "document_token_stats.json"

    # 讀取文檔
    doc_token_info = []

    print(f"Reading and tokenizing documents in {dataset_dir}...")
    for filename in tqdm(os.listdir(dataset_dir)):
        if filename.endswith(".txt"):
            doc_id = filename.replace(".txt", "")
            file_path = dataset_dir / filename

            with file_path.open('r', encoding='utf-8') as f:
                text = f.read().strip()

            token_count = len(tokenizer.encode(text, truncation=False))
            doc_token_info.append({
                "id": doc_id,
                "token_count": token_count
            })

    # 儲存 JSON
    with output_json.open('w', encoding='utf-8') as f:
        json.dump(doc_token_info, f, indent=2, ensure_ascii=False)

    print(f"已儲存 token 統計資訊到 {output_json}")


/home/lab/miniconda3/envs/THUIR-COLIEE2023-WSL/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Task1 year: 2026, dir: /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026
Reading and tokenizing documents in /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/processed...


100%|██████████| 7714/7714 [00:47<00:00, 163.63it/s]


已儲存 token 統計資訊到 /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/processed/document_token_stats.json
Reading and tokenizing documents in /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/processed_new...


100%|██████████| 5/5 [00:00<00:00, 100342.20it/s]


已儲存 token 統計資訊到 /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/processed_new/document_token_stats.json
Reading and tokenizing documents in /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/summary...


100%|██████████| 969/969 [00:00<00:00, 3644.08it/s]

已儲存 token 統計資訊到 /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/summary/document_token_stats.json


In [7]:
# 統計有多少文檔超過 0/1024/2048/4096/6144/8192/12288 tokens，以及平均 token 數量
import json
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "Legal Case Retrieval" / "lcr" / "task1_paths.py").exists():
            return path
    raise FileNotFoundError("Cannot locate repo root from current working directory")


if "task1_dir" not in globals():
    repo_root = find_repo_root(Path.cwd())
    package_root = repo_root / "Legal Case Retrieval"
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))

    from lcr.task1_paths import get_task1_dir, get_task1_year

    task1_dir = Path(get_task1_dir())
    task1_year = get_task1_year()
    print(f"Using Task1 year: {task1_year}, dir: {task1_dir}")


def analyze_token_stats(json_path: Path):
    if not json_path.exists():
        print(f"Skip missing stats json: {json_path}")
        return

    with json_path.open('r', encoding='utf-8') as f:
        doc_token_info = json.load(f)

    if not doc_token_info:
        print(f"Skip empty stats json: {json_path}")
        return

    total_docs = len(doc_token_info)
    count_0 = sum(1 for doc in doc_token_info if doc["token_count"] > 0)
    count_1024 = sum(1 for doc in doc_token_info if doc["token_count"] > 1024)
    count_2048 = sum(1 for doc in doc_token_info if doc["token_count"] > 2048)
    count_4096 = sum(1 for doc in doc_token_info if doc["token_count"] > 4096)
    count_6144 = sum(1 for doc in doc_token_info if doc["token_count"] > 6144)
    count_8192 = sum(1 for doc in doc_token_info if doc["token_count"] > 8192)
    count_12288 = sum(1 for doc in doc_token_info if doc["token_count"] > 12288)
    avg_tokens = sum(doc["token_count"] for doc in doc_token_info) / total_docs

    ratio_0 = count_0 / total_docs
    ratio_1024 = count_1024 / total_docs
    ratio_2048 = count_2048 / total_docs
    ratio_4096 = count_4096 / total_docs
    ratio_6144 = count_6144 / total_docs
    ratio_8192 = count_8192 / total_docs
    ratio_12288 = count_12288 / total_docs

    print(f"\n[{json_path.parent}]")
    print(f"Total documents: {total_docs}")
    print(f"Documents > 0 tokens: {count_0} ({ratio_0:.2%})")
    print(f"Documents > 1024 tokens: {count_1024} ({ratio_1024:.2%})")
    print(f"Documents > 2048 tokens: {count_2048} ({ratio_2048:.2%})")
    print(f"Documents > 4096 tokens: {count_4096} ({ratio_4096:.2%})")
    print(f"Documents > 6144 tokens: {count_6144} ({ratio_6144:.2%})")
    print(f"Documents > 8192 tokens: {count_8192} ({ratio_8192:.2%})")
    print(f"Documents > 12288 tokens: {count_12288} ({ratio_12288:.2%})")
    print(f"Average tokens per document: {avg_tokens:.2f}")


for folder_name in ["processed", "processed_new", "summary"]:
    analyze_token_stats(task1_dir / folder_name / "document_token_stats.json")


[/home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/processed]
Total documents: 7708
Documents > 0 tokens: 7708 (100.00%)
Documents > 1024 tokens: 7370 (95.61%)
Documents > 2048 tokens: 6121 (79.41%)
Documents > 4096 tokens: 3382 (43.88%)
Documents > 6144 tokens: 1703 (22.09%)
Documents > 8192 tokens: 949 (12.31%)
Documents > 12288 tokens: 401 (5.20%)
Average tokens per document: 5031.36
Skip empty stats json: /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/processed_new/document_token_stats.json

[/home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/summary]
Total documents: 967
Documents > 0 tokens: 967 (100.00%)
Documents > 1024 tokens: 0 (0.00%)
Documents > 2048 tokens: 0 (0.00%)
Documents > 4096 tokens: 0 (0.00%)
Documents > 6144 tokens: 0 (0.00%)
Documents > 8192 tokens: 0 (0.00%)
Documents > 12288 tokens: 0 (0.00%)
Average tokens per document: 132.62


## 畫統計圖

In [2]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

if "task1_dir" not in globals():
    import sys

    def find_repo_root(start: Path) -> Path:
        for path in [start, *start.parents]:
            if (path / "Legal Case Retrieval" / "lcr" / "task1_paths.py").exists():
                return path
        raise FileNotFoundError("Cannot locate repo root from current working directory")

    repo_root = find_repo_root(Path.cwd())
    package_root = repo_root / "Legal Case Retrieval"
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))

    from lcr.task1_paths import get_task1_dir, get_task1_year

    task1_dir = Path(get_task1_dir())
    task1_year = get_task1_year()

# ===== 可調參數 =====
bin_size = 513
max_token_display = 10000

# ===== 資料夾清單 =====
base_folders = [
    task1_dir / "processed",
    task1_dir / "processed_new",
    task1_dir / "summary",
]

for folder in base_folders:
    input_json = folder / "document_token_stats.json"
    output_plot = folder / "token_distribution_from_json.png"

    if not input_json.exists():
        print(f"Skip missing stats json: {input_json}")
        continue

    with input_json.open('r', encoding='utf-8') as f:
        doc_token_info = json.load(f)

    token_counts = [item["token_count"] for item in doc_token_info]

    if not token_counts:
        print(f"Skip empty stats json: {input_json}")
        continue

    # 計算平均
    average_tokens = sum(token_counts) / len(token_counts)
    print(f"[{folder}] Average token count: {average_tokens:.2f}")

    # 過濾顯示範圍內的 token 數
    filtered_token_counts = [t for t in token_counts if t <= max_token_display]

    # 分 bin 畫圖
    bins = list(range(0, (max_token_display // bin_size + 1) * bin_size + 1, bin_size))
    hist, bin_edges = np.histogram(filtered_token_counts, bins=bins)

    # 畫圖
    plt.figure(figsize=(10, 6))
    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])-1}" for i in range(len(hist))]
    plt.bar(labels, hist)
    plt.xticks(rotation=45)
    plt.xlabel("Token Count Range (≤ {} tokens)".format(max_token_display))
    plt.ylabel("Number of Documents")
    plt.title("Document Length (Token Count) Distribution")
    plt.tight_layout()
    plt.savefig(output_plot)
    print(f"圖表儲存為 {output_plot}")
    plt.close()


[/home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/processed] Average token count: 5031.36
圖表儲存為 /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/processed/token_distribution_from_json.png
Skip empty stats json: /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/processed_new/document_token_stats.json
[/home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/summary] Average token count: 132.62
圖表儲存為 /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026/summary/token_distribution_from_json.png


# 計算訓練及驗證資料分別查詢數量、每個查詢平均相關數

In [3]:
# 訓練資料有幾個query
from typing import List
from pathlib import Path
import pandas as pd


def load_query_ids(path: Path) -> List[str]:
    with path.open('r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


if "task1_dir" not in globals():
    import sys

    def find_repo_root(start: Path) -> Path:
        for path in [start, *start.parents]:
            if (path / "Legal Case Retrieval" / "lcr" / "task1_paths.py").exists():
                return path
        raise FileNotFoundError("Cannot locate repo root from current working directory")

    repo_root = find_repo_root(Path.cwd())
    package_root = repo_root / "Legal Case Retrieval"
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))

    from lcr.task1_paths import get_task1_dir, get_task1_year

    task1_dir = Path(get_task1_dir())
    task1_year = get_task1_year()

base_folder = Path(task1_dir)
print(f"Using Task1 year: {task1_year}, dir: {base_folder}")

# 訓練資料query數量
train_qid_path = base_folder / 'train_qid.tsv'
train_qids = load_query_ids(train_qid_path)
display("訓練資料query數量: " + str(len(train_qids)))

# 驗證資料query數量
valid_qid_path = base_folder / 'valid_qid.tsv'
valid_qids = load_query_ids(valid_qid_path)
display("驗證資料query數量: " + str(len(valid_qids)))

# 讀訓練及驗證的標註資料
import json
labels_path = base_folder / f'task1_train_labels_{task1_year}.json'
with labels_path.open('r', encoding='utf-8') as f:
    train_labels_json = json.load(f)

# ---訓練資料相關數量分析---
train_query_relevance_counts = {}
for qid in train_qids:
    qid_txt = qid + '.txt'
    if qid_txt not in train_labels_json:
        print(f"警告: query id {qid} 不在 {labels_path.name} 中")
    else:
        train_query_relevance_counts[qid] = len(train_labels_json[qid_txt])

if train_query_relevance_counts:
    avg_relevance = sum(train_query_relevance_counts.values()) / len(train_query_relevance_counts)
    print(f"訓練資料query的平均相關數量: {avg_relevance}")
else:
    print("訓練資料沒有有效的query相關數量")

import plotly.express as px
hist_data = list(train_query_relevance_counts.values())
fig = px.histogram(
    x=hist_data,
    title="訓練資料query的相關數量分佈",
    labels={"x": "相關數量", "y": "查詢數量"}
)
fig.update_xaxes(title="相關數量")
fig.update_yaxes(title="查詢數量")
fig.show()

# ---驗證資料相關數量分析---
valid_query_relevance_counts = {}
for qid in valid_qids:
    qid_txt = qid + '.txt'
    if qid_txt not in train_labels_json:
        print(f"警告: query id {qid} 不在 {labels_path.name} 中")
    else:
        valid_query_relevance_counts[qid] = len(train_labels_json[qid_txt])

if valid_query_relevance_counts:
    avg_relevance = sum(valid_query_relevance_counts.values()) / len(valid_query_relevance_counts)
    print(f"驗證資料query的平均相關數量: {avg_relevance}")
else:
    print("驗證資料沒有有效的query相關數量")

hist_data = list(valid_query_relevance_counts.values())
fig = px.histogram(
    x=hist_data,
    title="驗證資料query的相關數量分佈",
    labels={"x": "相關數量", "y": "查詢數量"}
)
fig.update_xaxes(title="相關數量")
fig.update_yaxes(title="查詢數量")
fig.show()


Using Task1 year: 2026, dir: /home/lab/THUIR-COLIEE2023/coliee_dataset/task1/2026


'訓練資料query數量: 1600'

'驗證資料query數量: 401'

訓練資料query的平均相關數量: 4.121875


驗證資料query的平均相關數量: 4.129675810473816
